In [8]:
import pyspark

In [9]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
        .master("local[*]")
        .appName("test4")
        .getOrCreate()
)

26/03/02 19:44:32 WARN Utils: Your hostname, Christian resolves to a loopback address: 127.0.1.1; using 172.24.63.94 instead (on interface eth0)
26/03/02 19:44:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/02 19:44:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [10]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/fhv_tripdata_2021-02.parquet

--2026-03-02 19:44:36--  https://d37ci6vzurychx.cloudfront.net/trip-data/fhv_tripdata_2021-02.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.172.223.187, 18.172.223.180, 18.172.223.91, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.172.223.187|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10645466 (10M) [binary/octet-stream]
Saving to: ‘fhv_tripdata_2021-02.parquet.1’

fhv_tripdata_2021-0 100%[===================>]  10.15M  11.7MB/s    in 0.9s    

2026-03-02 19:44:37 (11.7 MB/s) - ‘fhv_tripdata_2021-02.parquet.1’ saved [10645466/10645466]



In [11]:
df = spark.read \
   .option("header", "true") \
   .parquet('fhv_tripdata_2021-02.parquet')
df.show()

+--------------------+-------------------+-------------------+------------+------------+-------+----------------------+
|dispatching_base_num|    pickup_datetime|   dropOff_datetime|PUlocationID|DOlocationID|SR_Flag|Affiliated_base_number|
+--------------------+-------------------+-------------------+------------+------------+-------+----------------------+
|              B00013|2021-02-01 00:01:00|2021-02-01 01:33:00|        NULL|        NULL|   NULL|                B00014|
|     B00021         |2021-02-01 00:55:40|2021-02-01 01:06:20|       173.0|        82.0|   NULL|       B00021         |
|     B00021         |2021-02-01 00:14:03|2021-02-01 00:28:37|       173.0|        56.0|   NULL|       B00021         |
|     B00021         |2021-02-01 00:27:48|2021-02-01 00:35:45|        82.0|       129.0|   NULL|       B00021         |
|              B00037|2021-02-01 00:12:50|2021-02-01 00:26:38|        NULL|       225.0|   NULL|                B00037|
|              B00037|2021-02-01 00:00:3

In [12]:
df.head(5)

[Row(dispatching_base_num='B00013', pickup_datetime=datetime.datetime(2021, 2, 1, 0, 1), dropOff_datetime=datetime.datetime(2021, 2, 1, 1, 33), PUlocationID=None, DOlocationID=None, SR_Flag=None, Affiliated_base_number='B00014'),
 Row(dispatching_base_num='B00021         ', pickup_datetime=datetime.datetime(2021, 2, 1, 0, 55, 40), dropOff_datetime=datetime.datetime(2021, 2, 1, 1, 6, 20), PUlocationID=173.0, DOlocationID=82.0, SR_Flag=None, Affiliated_base_number='B00021         '),
 Row(dispatching_base_num='B00021         ', pickup_datetime=datetime.datetime(2021, 2, 1, 0, 14, 3), dropOff_datetime=datetime.datetime(2021, 2, 1, 0, 28, 37), PUlocationID=173.0, DOlocationID=56.0, SR_Flag=None, Affiliated_base_number='B00021         '),
 Row(dispatching_base_num='B00021         ', pickup_datetime=datetime.datetime(2021, 2, 1, 0, 27, 48), dropOff_datetime=datetime.datetime(2021, 2, 1, 0, 35, 45), PUlocationID=82.0, DOlocationID=129.0, SR_Flag=None, Affiliated_base_number='B00021         ')

In [10]:
df.schema

StructType([StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', TimestampNTZType(), True), StructField('dropOff_datetime', TimestampNTZType(), True), StructField('PUlocationID', DoubleType(), True), StructField('DOlocationID', DoubleType(), True), StructField('SR_Flag', IntegerType(), True), StructField('Affiliated_base_number', StringType(), True)])

In [19]:
out_dir = "fhv_tripdata_2021-02_csv_out" 

(df.coalesce(1)
   .write
   .mode("overwrite")
   .option("header", True)
   .csv(out_dir)
)

In [20]:
import os, glob, shutil

out_dir = "fhv_tripdata_2021-02_csv_out"
final_csv = "fhv_tripdata_2021-02.csv"

part_file = glob.glob(os.path.join(out_dir, "part-*.csv"))[0]
shutil.move(part_file, final_csv)
shutil.rmtree(out_dir)

print("OK ->", final_csv)

OK -> fhv_tripdata_2021-02.csv


In [21]:
!head -n 1001 fhv_tripdata_2021-02.csv > head.csv

In [22]:
import pandas as pd
pd.__version__

'3.0.1'

In [23]:
df_pandas = pd.read_csv('head.csv')

In [24]:
df_pandas.dtypes

dispatching_base_num          str
pickup_datetime               str
dropOff_datetime              str
PUlocationID              float64
DOlocationID              float64
SR_Flag                   float64
Affiliated_base_number        str
dtype: object

In [25]:
spark.createDataFrame(df_pandas).show()

[Stage 8:>                                                          (0 + 1) / 1]

+--------------------+--------------------+--------------------+------------+------------+-------+----------------------+
|dispatching_base_num|     pickup_datetime|    dropOff_datetime|PUlocationID|DOlocationID|SR_Flag|Affiliated_base_number|
+--------------------+--------------------+--------------------+------------+------------+-------+----------------------+
|              B00013|2021-02-01T00:01:...|2021-02-01T01:33:...|         NaN|         NaN|    NaN|                B00014|
|              B00021|2021-02-01T00:55:...|2021-02-01T01:06:...|       173.0|        82.0|    NaN|                B00021|
|              B00021|2021-02-01T00:14:...|2021-02-01T00:28:...|       173.0|        56.0|    NaN|                B00021|
|              B00021|2021-02-01T00:27:...|2021-02-01T00:35:...|        82.0|       129.0|    NaN|                B00021|
|              B00037|2021-02-01T00:12:...|2021-02-01T00:26:...|         NaN|       225.0|    NaN|                B00037|
|              B00037|20

In [26]:
spark.createDataFrame(df_pandas).schema

StructType([StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropOff_datetime', StringType(), True), StructField('PUlocationID', DoubleType(), True), StructField('DOlocationID', DoubleType(), True), StructField('SR_Flag', DoubleType(), True), StructField('Affiliated_base_number', StringType(), True)])

In [30]:
from pyspark.sql import types

In [35]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True),
    types.StructField('dropoff_datetime', types.TimestampType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('SR_Flag', types.StringType(), True)
])

In [38]:
df = spark.read \
   .option("header", "true") \
   .schema(schema) \
   .csv('fhv_tripdata_2021-02.csv')

In [40]:
df.head(10)

26/03/02 20:04:24 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: dispatching_base_num, pickup_datetime, dropOff_datetime, PUlocationID, DOlocationID, SR_Flag, Affiliated_base_number
 Schema: hvfhs_license_num, dispatching_base_num, pickup_datetime, dropoff_datetime, PULocationID, DOLocationID, SR_Flag
Expected: hvfhs_license_num but found: dispatching_base_num
CSV file: file:///mnt/c/Users/Christian/Documents/data_engineer_zoomcamp_2026/Module6/notebooks/fhv_tripdata_2021-02.csv


[Row(hvfhs_license_num='B00013', dispatching_base_num='2021-02-01T00:01:00.000', pickup_datetime=datetime.datetime(2021, 2, 1, 1, 33), dropoff_datetime=None, PULocationID=None, DOLocationID=None, SR_Flag='B00014'),
 Row(hvfhs_license_num='B00021', dispatching_base_num='2021-02-01T00:55:40.000', pickup_datetime=datetime.datetime(2021, 2, 1, 1, 6, 20), dropoff_datetime=None, PULocationID=None, DOLocationID=None, SR_Flag='B00021'),
 Row(hvfhs_license_num='B00021', dispatching_base_num='2021-02-01T00:14:03.000', pickup_datetime=datetime.datetime(2021, 2, 1, 0, 28, 37), dropoff_datetime=None, PULocationID=None, DOLocationID=None, SR_Flag='B00021'),
 Row(hvfhs_license_num='B00021', dispatching_base_num='2021-02-01T00:27:48.000', pickup_datetime=datetime.datetime(2021, 2, 1, 0, 35, 45), dropoff_datetime=None, PULocationID=None, DOLocationID=None, SR_Flag='B00021'),
 Row(hvfhs_license_num='B00037', dispatching_base_num='2021-02-01T00:12:50.000', pickup_datetime=datetime.datetime(2021, 2, 1, 0,

In [41]:
df = df.repartition(24)

In [42]:
df.write.parquet('fhvhv/2021/01/')

26/03/02 20:13:22 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: dispatching_base_num, pickup_datetime, dropOff_datetime, PUlocationID, DOlocationID, SR_Flag, Affiliated_base_number
 Schema: hvfhs_license_num, dispatching_base_num, pickup_datetime, dropoff_datetime, PULocationID, DOLocationID, SR_Flag
Expected: hvfhs_license_num but found: dispatching_base_num
CSV file: file:///mnt/c/Users/Christian/Documents/data_engineer_zoomcamp_2026/Module6/notebooks/fhv_tripdata_2021-02.csv
26/03/02 20:13:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/02 20:13:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/02 20:13:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/03/02 20:13:26 WARN

In [43]:
df = spark.read.parquet('fhvhv/2021/01/')

In [44]:
df

DataFrame[hvfhs_license_num: string, dispatching_base_num: string, pickup_datetime: timestamp, dropoff_datetime: timestamp, PULocationID: int, DOLocationID: int, SR_Flag: string]

In [45]:
df.show()

+-----------------+--------------------+-------------------+----------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+----------------+------------+------------+-------+
|           B01768|2021-02-02T07:01:...|2021-02-02 07:22:03|            NULL|        NULL|        NULL| B01768|
|           B01145|2021-02-02T13:54:...|2021-02-02 14:08:34|            NULL|        NULL|        NULL| B02879|
|           B01231|2021-02-01T09:19:...|2021-02-01 09:52:36|            NULL|        NULL|        NULL| B02764|
|           B00856|2021-02-03T11:47:...|2021-02-03 11:50:15|            NULL|        NULL|        NULL| B02836|
|           B00706|2021-02-03T08:24:...|2021-02-03 08:36:42|            NULL|        NULL|        NULL| B00706|
|           B03016|2021-02-03T14:26:...|2021-02-03 14:39:42|            NULL|        NULL|        NULL| 

In [46]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



In [54]:
df.select('pickup_datetime','dropoff_datetime','PULocationID','DOLocationID').filter(df.hvfhs_license_num == 'B01145').show()

+-------------------+----------------+------------+------------+
|    pickup_datetime|dropoff_datetime|PULocationID|DOLocationID|
+-------------------+----------------+------------+------------+
|2021-02-02 14:08:34|            NULL|        NULL|        NULL|
|2021-02-01 05:30:35|            NULL|        NULL|        NULL|
|2021-02-02 15:34:39|            NULL|        NULL|        NULL|
|2021-02-03 02:18:00|            NULL|        NULL|        NULL|
|2021-02-02 18:48:47|            NULL|        NULL|        NULL|
|2021-02-02 14:28:12|            NULL|        NULL|        NULL|
|2021-02-03 13:08:32|            NULL|        NULL|        NULL|
|2021-02-02 17:49:53|            NULL|        NULL|        NULL|
|2021-02-02 16:59:51|            NULL|        NULL|        NULL|
|2021-02-03 15:04:04|            NULL|        NULL|        NULL|
|2021-02-03 13:21:32|            NULL|        NULL|        NULL|
|2021-02-02 22:35:53|            NULL|        NULL|        NULL|
|2021-02-03 17:59:54|    

In [55]:
from pyspark.sql import functions as F

In [61]:
df \
.withColumn('pickup_date',F.to_date(df.pickup_datetime)) \
.withColumn( 'dropoff_date',F.to_date(df.dropoff_datetime)) \
.select('pickup_date','dropoff_date', 'PULocationID', 'DOLocationID') \
.show()

+-----------+------------+------------+------------+
|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-----------+------------+------------+------------+
| 2021-02-02|        NULL|        NULL|        NULL|
| 2021-02-02|        NULL|        NULL|        NULL|
| 2021-02-01|        NULL|        NULL|        NULL|
| 2021-02-03|        NULL|        NULL|        NULL|
| 2021-02-03|        NULL|        NULL|        NULL|
| 2021-02-03|        NULL|        NULL|        NULL|
| 2021-02-02|        NULL|        NULL|        NULL|
| 2021-02-03|        NULL|        NULL|        NULL|
| 2021-02-02|        NULL|        NULL|        NULL|
| 2021-02-01|        NULL|        NULL|        NULL|
| 2021-02-01|        NULL|        NULL|        NULL|
| 2021-02-01|        NULL|        NULL|        NULL|
| 2021-02-02|        NULL|        NULL|        NULL|
| 2021-02-01|        NULL|        NULL|        NULL|
| 2021-02-01|        NULL|        NULL|        NULL|
| 2021-02-01|        NULL|        NULL|       

In [62]:
def crazzy_stuff(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f's/{num:03x}'
    elif num % 3 == 0:
        return f'a/{num:03x}'
    else:
        return f'e/{num:03x}'

In [64]:
crazzy_stuff('B02884')

's/b44'

In [66]:
crazy_stuff_udf = F.udf(crazzy_stuff, returnType = types.StringType())

In [68]:
df \
.withColumn('pickup_date',F.to_date(df.pickup_datetime)) \
.withColumn( 'dropoff_date',F.to_date(df.dropoff_datetime)) \
.withColumn('base_id',crazy_stuff_udf(df.hvfhs_license_num)) \
.select('base_id','pickup_date','dropoff_date', 'PULocationID', 'DOLocationID') \
.show()

+-------+-----------+------------+------------+------------+
|base_id|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-------+-----------+------------+------------+------------+
|  e/6e8| 2021-02-02|        NULL|        NULL|        NULL|
|  e/479| 2021-02-02|        NULL|        NULL|        NULL|
|  e/4cf| 2021-02-01|        NULL|        NULL|        NULL|
|  e/358| 2021-02-03|        NULL|        NULL|        NULL|
|  e/2c2| 2021-02-03|        NULL|        NULL|        NULL|
|  e/bc8| 2021-02-03|        NULL|        NULL|        NULL|
|  e/4cf| 2021-02-02|        NULL|        NULL|        NULL|
|  e/b09| 2021-02-03|        NULL|        NULL|        NULL|
|  s/8ce| 2021-02-02|        NULL|        NULL|        NULL|
|  a/9f6| 2021-02-01|        NULL|        NULL|        NULL|
|  a/9f6| 2021-02-01|        NULL|        NULL|        NULL|
|  e/4cf| 2021-02-01|        NULL|        NULL|        NULL|
|  s/015| 2021-02-02|        NULL|        NULL|        NULL|
|  e/6b5| 2021-02-01|   